# ShelfPulse AI — YOLO11n SKU-110K Retail Training & NCNN Vulkan Export
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tahadeab-ai/shelf-pulse-ai/blob/main/notebooks/train_yolo11_sku110k_colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/tahadeab-ai/shelf-pulse-ai)
[![License](https://img.shields.io/badge/License-Apache_2.0-green.svg)](https://opensource.org/licenses/Apache-2.0)

**Project:** ShelfPulse AI (`shelf-pulse-ai`)  
**Target:** Android Edge-AI Real-Time Retail Shelf Inventory Counting  
**Model:** YOLO11n (FP16 NCNN with Vulkan GPU acceleration & ByteTrack-cpp)  
**Dataset:** SKU-110K (Dense retail products, 110,000+ images/shelves)

> **Note on Licensing:** The Android application is licensed under Apache-2.0. This training pipeline utilizes Ultralytics YOLO11 which is licensed under AGPL-3.0.


## 1. Environment Setup & GPU Verification
Verify that a CUDA-capable GPU (T4, V100, or A100) is allocated, install Ultralytics, PyTorch, and NCNN conversion tools (`pnnx`).


In [ ]:
!nvidia-smi

# Install Ultralytics and dependencies
!pip install -q --upgrade ultralytics onnx onnxslim onnxruntime
!pip install -q pnnx

import torch
import ultralytics
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
ultralytics.checks()


## 2. Optional: Mount Google Drive
Mount Google Drive to persist training checkpoints and exported NCNN weights across Colab timeouts.


In [ ]:
import os
from google.colab import drive

MOUNT_DRIVE = False  # Set to True to persist to Google Drive

if MOUNT_DRIVE:
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/ShelfPulse_Checkpoints'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print("Checkpoints will be saved to:", CHECKPOINT_DIR)
else:
    CHECKPOINT_DIR = '/content/runs'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print("Using local Colab storage.")


## 3. Dataset Configuration: SKU-110K (or Custom Roboflow)
The SKU-110K dataset is specifically designed for dense object detection on retail shelves.
By default, Ultralytics downloads and parses `SKU-110K.yaml` automatically.
You can also swap in your custom Roboflow retail dataset.


In [ ]:
# Configuration block
DATASET_SOURCE = "SKU-110K" # Options: "SKU-110K" or "ROBOFLOW"

if DATASET_SOURCE == "SKU-110K":
    DATA_YAML = "SKU-110K.yaml"
    print("Using standardized dense retail dataset:", DATA_YAML)
elif DATASET_SOURCE == "ROBOFLOW":
    # Example Roboflow download template:
    # !pip install -q roboflow
    # from roboflow import Roboflow
    # rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
    # project = rf.workspace("retail").project("shelf-products")
    # version = project.version(1)
    # dataset = version.download("yolov11")
    # DATA_YAML = f"{dataset.location}/data.yaml"
    pass


## 4. Model Training: YOLO11n with Retail Shelf Hyperparameters
We initialize from pretrained `yolo11n.pt` (Nano model with ~2.6M params, optimized for mobile 60 FPS inference).
Key augmentations tailored for retail shelves:
- `mosaic=1.0`: Crucial for learning dense object packing.
- `scale=0.5`: Robustness to varying distance from shelf.
- `hsv_h=0.015, hsv_s=0.7, hsv_v=0.4`: Robustness to varied store lighting.


In [ ]:
from ultralytics import YOLO

# Load lightweight YOLO11 Nano model
model = YOLO("yolo11n.pt")

# Execute training with retail-tuned parameters
results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=32,            # Adjust according to available GPU VRAM (32 for T4)
    workers=4,
    device=0,
    optimizer="AdamW",
    lr0=0.002,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    patience=10,         # Early stopping patience
    project="shelfpulse_sku110k",
    name="yolo11n_run",
    save=True,
    exist_ok=True
)


## 5. Model Evaluation & Benchmark Metrics
Compute mAP@50, mAP@50-95, Precision, and Recall. Export performance plots into `eval_artifacts/`.


In [ ]:
import glob
import shutil
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Validate the best trained checkpoint
val_model = YOLO("shelfpulse_sku110k/yolo11n_run/weights/best.pt")
metrics = val_model.val(data=DATA_YAML, imgsz=640, batch=16, split="val")

print("\n====== MODEL EVALUATION RESULTS ======")
print(f"mAP@50:     {metrics.box.map50:.4f}")
print(f"mAP@50-95:  {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")
print("=======================================\n")

# Create directory for documentation artifacts
os.makedirs("eval_artifacts", exist_ok=True)
plot_files = ["confusion_matrix.png", "results.png", "PR_curve.png", "val_batch0_pred.jpg"]
run_dir = "shelfpulse_sku110k/yolo11n_run"

for f in plot_files:
    src = os.path.join(run_dir, f)
    if os.path.exists(src):
        dst = os.path.join("eval_artifacts", f)
        shutil.copy2(src, dst)
        print(f"Copied metric plot: {dst}")


## 6. Export to NCNN (FP16 Vulkan Optimized)
Convert PyTorch weights to NCNN FP16 format for native C++ execution on Android.
This creates `model.ncnn.param` and `model.ncnn.bin` with layer names matching our C++ JNI engine (`in0`, `out0`).


In [ ]:
# Export to NCNN FP16
exported_path = val_model.export(
    format="ncnn",
    imgsz=640,
    half=True  # FP16 precision for 2x mobile inference speed
)

print(f"NCNN Model exported to: {exported_path}")

# Verify exported files
import hashlib

export_dir = "shelfpulse_sku110k/yolo11n_run/weights/best_ncnn_model"
param_file = os.path.join(export_dir, "model.ncnn.param")
bin_file = os.path.join(export_dir, "model.ncnn.bin")

assert os.path.exists(param_file), "model.ncnn.param missing!"
assert os.path.exists(bin_file), "model.ncnn.bin missing!"

def calc_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

param_hash = calc_sha256(param_file)
bin_hash = calc_sha256(bin_file)

with open(os.path.join(export_dir, "SHA256SUMS.txt"), "w") as f:
    f.write(f"{param_hash}  model.ncnn.param\n")
    f.write(f"{bin_hash}  model.ncnn.bin\n")

print("Generated SHA256SUMS.txt:")
print(f"{param_hash}  model.ncnn.param")
print(f"{bin_hash}  model.ncnn.bin")

# Package into distribution zip
shutil.make_archive("shelfpulse_yolo11n_sku110k_ncnn", "zip", export_dir)
print("Created release package: shelfpulse_yolo11n_sku110k_ncnn.zip")


## 7. Publish Weights to GitHub Releases
Upload the trained NCNN weights directly to GitHub Releases under `models-v1.0.0`.


In [ ]:
GITHUB_REPO = "tahadeab-ai/shelf-pulse-ai"
RELEASE_TAG = "models-v1.0.0"

print(f"Target repository: https://github.com/{GITHUB_REPO}/releases/tag/{RELEASE_TAG}")
print("\nTo publish via GitHub CLI:")
print(f"!gh release create {RELEASE_TAG} shelfpulse_yolo11n_sku110k_ncnn.zip --repo {GITHUB_REPO} --title 'Trained YOLO11n SKU-110K NCNN Weights' --notes 'NCNN FP16 weights trained on SKU-110K dataset.'")
